Stage 2c — Backtest audit

The `backtest-auditor` subagent's checklist, run as code rather than asserted
in prose. It reads what the forecasting stage wrote and re-derives the claims
from the artefacts, so a mismatch between the notebook and the JSON shows up
here instead of in a reader's head.

Read-only by design: this script asserts and reports, it never repairs. A
harness that fails audit is re-run, not patched from here.

The checks are structural rather than textual. Grepping source for suspicious
words finds the word, not the defect; re-deriving the origin counts from the
frame finds the defect.

In [2]:
import json
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)

REPORTS, METRICS, PROCESSED = "results/reports", "results/metrics", "datasets/processed"
TEST_START = pd.Timestamp("2026-02-24")
PURGE = 10

records = json.load(open(f"{METRICS}/forecast_results.json"))
frame = pd.read_csv(f"{PROCESSED}/model_frame.csv", parse_dates=["date"]).set_index("date")
print("records:", len(records), " frame:", frame.shape)

blockers, notes, checked, unchecked = [], [], [], []

records: 30  frame: (264, 72)


1. Split integrity

In [4]:
ycols = [c for c in frame.columns if c.startswith("h") and c[1:].isdigit()]
idx = frame.index
origins = [d for d in idx if d >= TEST_START]
usable = [o for o in origins if idx.get_loc(o) - PURGE >= 120]
print(f"origins at or after {TEST_START.date()}: {len(origins)}   usable: {len(usable)}")

first = usable[0]
train_end = idx[idx.get_loc(first) - PURGE - 1]
gap = idx.get_loc(first) - idx.get_loc(train_end) - 1
print(f"first origin {first.date()}  last training row {train_end.date()}  purged rows {gap}")
if train_end >= first:
    blockers.append("training window ends at or after the first forecast origin")
elif gap < PURGE:
    blockers.append(f"purge gap is {gap} rows, below the declared {PURGE}")
else:
    checked.append("1 split integrity: training ends before the origin with the full purge gap")

origins at or after 2026-02-24: 72   usable: 72
first origin 2026-02-24  last training row 2026-02-09  purged rows 10


2. Same test window across models

In [6]:
all_rows = [r for r in records if r.get("regime") == "all"]
by_h = {}
for r in all_rows:
    by_h.setdefault(r["horizon"], {})[r["model"]] = r.get("n")
ok_window = True
for h, models in sorted(by_h.items()):
    counts = set(models.values())
    print(f"h={h}: " + ", ".join(f"{m}={n}" for m, n in models.items()))
    if len(counts) > 1:
        ok_window = False
        blockers.append(f"h={h}: models scored on differing origin counts {counts}")
if ok_window:
    checked.append("2 benchmark contamination: every model scored on identical origins")

h=1: naive_random_walk=72, drift=72, elasticnet=72, random_forest=72, gradient_boosting=72
h=5: naive_random_walk=72, drift=72, elasticnet=72, random_forest=72, gradient_boosting=72


3. Benchmark present and first

In [8]:
models_present = [r["model"] for r in all_rows]
if not any("naive" in m for m in models_present):
    blockers.append("no naive benchmark in the results")
else:
    checked.append("3 benchmark present in the results table")
    if "naive" not in models_present[0]:
        notes.append("the benchmark is present but is not the first row of the table")

4. Unit consistency

RMSE in log-return units and in USD/bbl must be derivable from the same
predictions. If the two were computed in separate places they can drift apart,
and a ranking that differs between them is the symptom.

In [10]:
px = pd.read_csv(f"{PROCESSED}/master_daily.csv", parse_dates=["date"]).set_index("date")["brent_close"]
level = float(px.reindex(usable).mean())
consistent = True
for r in all_rows:
    if r.get("rmse_usd") is None or r.get("rmse_logret") is None:
        continue
    implied = float(r["rmse_logret"]) * level
    ratio = float(r["rmse_usd"]) / implied if implied else np.nan
    if not (0.7 < ratio < 1.4):
        consistent = False
        notes.append(f"{r['model']} h{r['horizon']}: USD and log-return RMSE differ by "
                     f"a factor of {ratio:.2f} from the first-order approximation")
print(f"mean price over the test block: {level:.2f}")
print("unit consistency within the first-order approximation:", consistent)
if consistent:
    checked.append("4 unit consistency: both RMSE unit systems agree to first order")

mean price over the test block: 83.69
unit consistency within the first-order approximation: True


5. Sample-size claims

In [12]:
size_ok = True
for r in all_rows:
    if r.get("n") and r["n"] > len(usable):
        size_ok = False
        blockers.append(f"{r['model']} claims n={r['n']} against {len(usable)} usable origins")
print(f"usable origins {len(usable)}; largest reported n "
      f"{max((r.get('n') or 0) for r in all_rows)}")
if size_ok:
    checked.append("5 sample-size claims match the number of usable origins")

ess_path = f"{REPORTS}/effective_sample.csv"
if os.path.exists(ess_path):
    checked.append("6 effective sample stated for the forward-filled features")
else:
    notes.append("effective sample size is not stated for the forward-filled ACLED features; "
                 "any standard error computed on origins alone is optimistic")

usable origins 72; largest reported n 72


6. What could not be checked

In [14]:
unchecked.append("3 feature construction: this script sees the built frame, not the code that "
                 "built it, so a leaking transform inside feature construction would not appear here")
if not any(r.get("regime") == "non_conflict" for r in records):
    unchecked.append("regime performance: no non-conflict slice was produced, so the split "
                     "could not be verified")

In [15]:
verdict = "FAIL" if blockers else ("PASS WITH NOTES" if notes else "PASS")
lines = [f"VERDICT: {verdict}"]
lines.append("BLOCKERS:  " + ("none" if not blockers else ""))
for b in blockers:
    lines.append("           " + b)
lines.append("NOTES:     " + ("none" if not notes else ""))
for n in notes:
    lines.append("           " + n)
lines.append("CHECKED:   " + ("; ".join(checked) if checked else "none"))
lines.append("UNCHECKED: " + ("; ".join(unchecked) if unchecked else "none"))

for line in lines:
    print(line)

with open(f"{REPORTS}/backtest_audit.txt", "w") as fh:
    for line in lines:
        print(line, file=fh)
print()
print("wrote", f"{REPORTS}/backtest_audit.txt")

VERDICT: PASS WITH NOTES
BLOCKERS:  none
NOTES:     
           effective sample size is not stated for the forward-filled ACLED features; any standard error computed on origins alone is optimistic
CHECKED:   1 split integrity: training ends before the origin with the full purge gap; 2 benchmark contamination: every model scored on identical origins; 3 benchmark present in the results table; 4 unit consistency: both RMSE unit systems agree to first order; 5 sample-size claims match the number of usable origins
UNCHECKED: 3 feature construction: this script sees the built frame, not the code that built it, so a leaking transform inside feature construction would not appear here

wrote results/reports/backtest_audit.txt
